<a href="https://colab.research.google.com/github/Wolfang-Perez/proyecto-integrador-pc-grupo-3/blob/main/notebooks/analisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto integrador — Grupo 3 · Logística
## Entregas de última milla en el Valle de Aburrá (sep–dic 2025)

**Curso:** Programación científica
**Grupo:** 3 — Campo: Logística
**Integrantes:** [Nombre completo 1] · [Nombre completo 2]
**Fecha:** noviembre de 2026
**Enlace al notebook en Colab:** [pegar aquí el enlace]


## 1. Carga e inspección de datos

En esta sección cargamos el CSV, revisamos su tamaño, las primeras filas, los tipos de cada columna y las estadísticas descriptivas. Esto nos permite conocer el dataset antes de limpiarlo y saber qué columnas hay que corregir.

**Nota de lectura:** siempre usamos **rutas relativas** (la guía lo exige). La celda busca el archivo en las tres ubicaciones habituales: la carpeta `data/` del repositorio, la carpeta `data/` un nivel arriba (al abrir el notebook desde `entregable-1/`) y la misma carpeta del notebook (cuando en Colab subes el CSV junto al `.ipynb`). Así funciona igual en tu computador y en Colab, sin tocar nada.


In [6]:
import os
import pandas as pd
import numpy as np

# Rutas relativas: el notebook funciona desde la raíz del repo, desde su carpeta o en Colab.
CANDIDATOS = [
    "data/logistica_entregas_valle_aburra_2025.csv",
    "../data/logistica_entregas_valle_aburra_2025.csv",
    "logistica_entregas_valle_aburra_2025.csv",
]
RUTA_CSV = next(r for r in CANDIDATOS if os.path.exists(r))

df = pd.read_csv(RUTA_CSV)

print("Archivo leído:", RUTA_CSV)
print("Forma (filas, columnas):", df.shape)
df.head()


Archivo leído: logistica_entregas_valle_aburra_2025.csv
Forma (filas, columnas): (4100, 21)


,id_pedido,fecha,dia_semana,franja_horaria,municipio_destino,canal_compra,categoria_producto,peso_kg,distancia_km,tipo_vehiculo,...,paradas_en_ruta,lluvia,tiempo_estimado_min,tiempo_real_min,retraso_min,estado_entrega,motivo_no_entrega,costo_envio_cop,calificacion_cliente,propina_cop
0,PED-00001,2025-09-01,lunes,Tarde,Medellín,NaN,Electrónica,1.25,NaN,Bicicleta eléctrica,...,17,No,68.0,80.0,12.0,A tiempo,NaN,6800.0,4.0,NaN
1,PED-00002,2025-09-01,lunes,Tarde,Copacabana,App,Electrónica,1.75,27.0,Moto,...,17,No,120.0,157.0,37.0,Con retraso,NaN,21800.0,NaN,NaN
2,PED-00003,2025-09-01,lunes,Tarde,Envigado,WhatsApp,Ropa,2.91,NaN,Moto,...,14,No,80.0,101.0,21.0,Con retraso,NaN,14200.0,5.0,1000.0
3,PED-00004,2025-09-01,lunes,Mañana,Girardota,App,Electrónica,2.25,21.2,Moto,...,16,No,103.0,131.0,28.0,Con retraso,NaN,17500.0,2.0,NaN
4,PED-00005,2025-09-01,lunes,Tarde,Medellín,App,Alimentos,7.28,6.2,Moto,...,10,Sí,50.0,91.0,41.0,Con retraso,NaN,10800.0,3.0,NaN


**Interpretación:** el dataset tiene **4.100 filas y 21 columnas**. Cada fila es **un pedido** entregado por uno de cuatro operadores logísticos en el Valle de Aburrá. `head()` muestra que hay columnas categóricas (`municipio_destino`, `canal_compra`, `tipo_vehiculo`, `operador`…), numéricas (`peso_kg`, `distancia_km`, `tiempo_*`, `costo_envio_cop`) y varias con celdas vacías (`canal_compra`, `distancia_km`, `calificacion_cliente`…), que analizaremos en el diagnóstico.


In [7]:
df.info()
print("\n--- dtypes ---")
print(df.dtypes)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_pedido             4100 non-null   object 
 1   fecha                 4100 non-null   object 
 2   dia_semana            4100 non-null   object 
 3   franja_horaria        4100 non-null   object 
 4   municipio_destino     4100 non-null   object 
 5   canal_compra          3877 non-null   object 
 6   categoria_producto    4100 non-null   object 
 7   peso_kg               3853 non-null   float64
 8   distancia_km          3988 non-null   float64
 9   tipo_vehiculo         4100 non-null   object 
 10  operador              4100 non-null   object 
 11  paradas_en_ruta       4100 non-null   int64  
 12  lluvia                4100 non-null   object 
 13  tiempo_estimado_min   4100 non-null   float64
 14  tiempo_real_min       3891 non-null   float64
 15  retraso_min          

**Interpretación:** `info()` confirma 21 columnas. La mayoría de los faltantes están en `calificacion_cliente`, `propina_cop` y `motivo_no_entrega`, lo cual es esperable (no todos califican, no todos dan propina y el motivo solo aplica si la entrega falla). El problema a corregir es `fecha`, que pandas leyó como texto (`object`) en lugar de fecha.


**¿Por qué convertir `fecha`?** Para poder filtrar por rangos de fechas (Black Friday, diciembre), agrupar por día o mes y graficar series temporales en el Entregable 2. Una fecha como texto no se puede ordenar ni agrupar correctamente.


In [8]:
# 'fecha' se leyó como texto (object); la convertimos a datetime.
df["fecha"] = pd.to_datetime(df["fecha"])

print("Nuevo tipo de 'fecha':", df["fecha"].dtype)
print("Rango de fechas:", df["fecha"].min().date(), "→", df["fecha"].max().date())


Nuevo tipo de 'fecha': datetime64[ns]
Rango de fechas: 2025-09-01 → 2025-12-31


In [9]:
df.describe(include="number").T


,count,mean,std,min,25%,50%,75%,max
peso_kg,3853.0,5.112702,23.736599,0.17,1.43,3.17,5.96,999.0
distancia_km,3988.0,11.271615,11.529938,0.00,6.10,8.70,13.30,480.0
paradas_en_ruta,4100.0,13.507561,6.292628,3.00,8.00,13.00,19.00,24.0
tiempo_estimado_min,4100.0,73.916585,26.871599,28.00,55.00,71.00,87.00,270.0
tiempo_real_min,3891.0,96.276022,44.113331,21.00,64.00,88.00,119.00,481.0
retraso_min,3891.0,22.220766,27.355507,-64.00,4.00,17.00,36.00,239.0
costo_envio_cop,4100.0,14025.829268,6261.358445,4600.00,9600.00,12100.00,17100.00,57800.0
calificacion_cliente,2992.0,3.360628,1.190046,1.00,3.00,4.00,4.00,5.0
propina_cop,927.0,1321.467098,1690.464556,0.00,0.00,0.00,2000.00,5000.0


**Interpretación de `describe()`:** los promedios y medianas se ven razonables (peso ≈ 5 kg, distancia ≈ 11 km, retraso medio ≈ 22 min), pero los **máximos** revelan valores imposibles: `peso_kg = 999`, `distancia_km = 480` y `retraso_min = 239`. Son candidatos a códigos centinela o errores de digitación que trataremos en la limpieza. También hay `retraso_min` negativo (hasta −64), que es válido: significa que el pedido llegó **antes** de lo prometido.


In [10]:
columnas_categoricas = [
    "dia_semana", "franja_horaria", "municipio_destino", "canal_compra",
    "categoria_producto", "tipo_vehiculo", "operador", "lluvia",
    "estado_entrega", "motivo_no_entrega",
]

for col in columnas_categoricas:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))



--- dia_semana ---
dia_semana
lunes        702
martes       671
miércoles    651
jueves       623
sábado       601
viernes      597
domingo      255
Name: count, dtype: int64

--- franja_horaria ---
franja_horaria
Tarde     1714
Mañana    1648
Noche      738
Name: count, dtype: int64

--- municipio_destino ---
municipio_destino
Medellín       1750
Bello           553
Envigado        408
Itagüí          393
Sabaneta        269
Copacabana      196
La Estrella     180
Caldas          133
Girardota       132
Barbosa          86
Name: count, dtype: int64

--- canal_compra ---
canal_compra
App         1746
Web         1192
WhatsApp     690
Teléfono     249
NaN          223
Name: count, dtype: int64

--- categoria_producto ---
categoria_producto
Alimentos      1180
Ropa            915
Hogar           746
Electrónica     630
Farmacia        629
Name: count, dtype: int64

--- tipo_vehiculo ---
tipo_vehiculo
Moto                   3155
Van                     597
Bicicleta eléctrica     304
Cam

**Interpretación:** las categóricas están bien distribuidas y sin categorías raras evidentes. Dato clave: `estado_entrega` tiene 2.001 "Con retraso" y 1.791 "A tiempo", y solo 308 fallidos/devueltos. `lluvia` está casi equilibrada (2.479 "No" vs 1.621 "Sí"). `operador` está desbalanceado: A atiende 1.299 pedidos y D solo 667, algo que hay que tener en cuenta al comparar operadores.


## 2. Diagnóstico de calidad de los datos

Aquí cuantificamos los faltantes (distinguiendo los reales de los que significan "no aplica"), los valores imposibles o centinela, y verificamos duplicados e inconsistencias.


In [11]:
faltantes = df.isna().sum()
porcentaje = (faltantes / len(df) * 100).round(2)

tabla_faltantes = pd.DataFrame({
    "faltantes": faltantes,
    "porcentaje": porcentaje,
}).sort_values("faltantes", ascending=False)

print(tabla_faltantes[tabla_faltantes["faltantes"] > 0])


                      faltantes  porcentaje
motivo_no_entrega          3792       92.49
propina_cop                3173       77.39
calificacion_cliente       1108       27.02
peso_kg                     247        6.02
canal_compra                223        5.44
retraso_min                 209        5.10
tiempo_real_min             209        5.10
distancia_km                112        2.73


**Interpretación de los faltantes.** Hay dos grupos:

1. **Faltantes "no aplica"** (no se imputan porque el vacío tiene significado):
   - `motivo_no_entrega` (92,49 %): vacío = el pedido **sí se entregó**.
   - `propina_cop` (77,39 %) y `calificacion_cliente` (27,02 %): vacío = el cliente **no** dejó propina / **no** calificó.
   - `tiempo_real_min` y `retraso_min` (5,10 % cada uno): vacío = la entrega **falló**.

2. **Faltantes reales** (sí se imputan): `peso_kg` (6,02 %), `canal_compra` (5,44 %) y `distancia_km` (2,73 %).


In [12]:
print("=== Valores sospechosos / centinela ===")

print("\npeso_kg mayor a 100 kg:")
print(df.loc[df["peso_kg"] > 100, ["id_pedido", "peso_kg", "tipo_vehiculo", "categoria_producto"]])

print("\ndistancia_km igual a 0 o mayor a 100 km:")
print(df.loc[(df["distancia_km"] == 0) | (df["distancia_km"] > 100),
             ["id_pedido", "distancia_km", "municipio_destino", "tipo_vehiculo"]])


=== Valores sospechosos / centinela ===

peso_kg mayor a 100 kg:
      id_pedido  peso_kg tipo_vehiculo categoria_producto
101   PED-00102    999.0           Van              Hogar
821   PED-00822    350.0          Moto               Ropa
1125  PED-01126    999.0          Moto           Farmacia

distancia_km igual a 0 o mayor a 100 km:
      id_pedido  distancia_km municipio_destino tipo_vehiculo
1536  PED-01537         250.0          Sabaneta          Moto
2178  PED-02179         480.0            Caldas          Moto
3168  PED-03169           0.0            Itagüí           Van


**Interpretación:** encontramos valores imposibles para la última milla:

- `peso_kg = 999` en 2 filas y `peso_kg = 350` en 1 fila de **Moto** (una moto no carga 350 kg) → centinelas.
- `distancia_km = 250` y `480` (el Valle de Aburrá mide ~70 km de punta a punta) y `distancia_km = 0` (la bodega no puede estar a 0 km del cliente) → imposibles.

Los convertiremos en `NaN` y luego los imputaremos con criterio, para no dejarlos inflando los promedios.


In [13]:
print("Filas duplicadas:", df.duplicated().sum())
print("id_pedido duplicados:", df["id_pedido"].duplicated().sum())

nombres_dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
dia_calculado = df["fecha"].dt.day_of_week.map(lambda i: nombres_dias[i])
print("Inconsistencias fecha↔dia_semana:", (df["dia_semana"] != dia_calculado).sum())

con_datos = df.dropna(subset=["tiempo_real_min", "tiempo_estimado_min", "retraso_min"])
diferencia = (con_datos["tiempo_real_min"] - con_datos["tiempo_estimado_min"] - con_datos["retraso_min"]).abs()
print("Filas donde retraso_min != real - estimado:", (diferencia > 0.01).sum())

fallidos = df[df["estado_entrega"] == "Fallido"]
print("Pedidos Fallido con tiempo_real no vacío:", fallidos["tiempo_real_min"].notna().sum())
print("Pedidos Fallido/Devuelto con calificación:",
      len(df[df["estado_entrega"].isin(["Fallido", "Devuelto"]) & df["calificacion_cliente"].notna()]))


Filas duplicadas: 0
id_pedido duplicados: 0
Inconsistencias fecha↔dia_semana: 0
Filas donde retraso_min != real - estimado: 0
Pedidos Fallido con tiempo_real no vacío: 0
Pedidos Fallido/Devuelto con calificación: 243


**Interpretación:** no hay filas duplicadas ni `id_pedido` repetidos; `fecha` y `dia_semana` son coherentes (0 inconsistencias); `retraso_min` equivale siempre a `tiempo_real_min − tiempo_estimado_min`; y los 209 pedidos `Fallido` no tienen `tiempo_real_min` (correcto). Hallazgo curioso: **243 pedidos `Fallido`/`Devuelto` sí tienen `calificacion_cliente`** — lo documentamos como inconsistencia semántica, pero no lo borramos porque puede ser el cliente calificando un intento fallido.


In [14]:
print("=== Inconsistencias de escritura en variables categóricas ===")
categoricas_texto = ["municipio_destino", "canal_compra", "categoria_producto",
                     "tipo_vehiculo", "operador", "estado_entrega",
                     "motivo_no_entrega", "dia_semana", "franja_horaria", "lluvia"]
for col in categoricas_texto:
    serie = df[col].dropna().astype(str)
    con_espacios = (serie != serie.str.strip()).sum()
    normalizada = serie.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    variantes = serie.nunique() - normalizada.nunique()
    print(f"{col}: categorías={serie.nunique()}, con espacios extra={con_espacios}, "
          f"variantes por mayúsculas/espacios={variantes}")


=== Inconsistencias de escritura en variables categóricas ===
municipio_destino: categorías=10, con espacios extra=0, variantes por mayúsculas/espacios=0
canal_compra: categorías=4, con espacios extra=0, variantes por mayúsculas/espacios=0
categoria_producto: categorías=5, con espacios extra=0, variantes por mayúsculas/espacios=0
tipo_vehiculo: categorías=4, con espacios extra=0, variantes por mayúsculas/espacios=0
operador: categorías=4, con espacios extra=0, variantes por mayúsculas/espacios=0
estado_entrega: categorías=4, con espacios extra=0, variantes por mayúsculas/espacios=0
motivo_no_entrega: categorías=7, con espacios extra=0, variantes por mayúsculas/espacios=0
dia_semana: categorías=7, con espacios extra=0, variantes por mayúsculas/espacios=0
franja_horaria: categorías=3, con espacios extra=0, variantes por mayúsculas/espacios=0
lluvia: categorías=2, con espacios extra=0, variantes por mayúsculas/espacios=0


**Interpretación:** buscamos categorías escritas de varias formas (espacios al inicio o al final, mayúsculas mezcladas, espacios dobles), porque `value_counts` las contaría como grupos distintos. En las diez categóricas el resultado es **0 espacios extra y 0 variantes**: el texto está normalizado y no hay nada que unificar.


In [15]:
print("=== ¿Por qué conservar motivo_no_entrega? ===")
print("Vacíos:", df["motivo_no_entrega"].isna().sum(), "de", len(df))
print("\nMotivo según estado de entrega:")
print(pd.crosstab(df["estado_entrega"], df["motivo_no_entrega"].fillna("(vacío)")))


=== ¿Por qué conservar motivo_no_entrega? ===
Vacíos: 3792 de 4100

Motivo según estado de entrega:
motivo_no_entrega  (vacío)  Cliente ausente  Cliente rechazó  \
estado_entrega                                                 
A tiempo              1791                0                0   
Con retraso           2001                0                0   
Devuelto                 0                0               31   
Fallido                  0              102                0   

motivo_no_entrega  Dirección incorrecta  Pedido equivocado  Producto dañado  \
estado_entrega                                                                
A tiempo                              0                  0                0   
Con retraso                           0                  0                0   
Devuelto                              0                 33               35   
Fallido                              49                  0                0   

motivo_no_entrega  Vehículo averiado  Zo

**Interpretación:** los **3.792 vacíos** de `motivo_no_entrega` coinciden exactamente con los **3.792 pedidos entregados** (1.791 "A tiempo" + 2.001 "Con retraso"): el vacío no es un dato perdido, es "no aplica". Entre las **308 entregas no exitosas** sí hay motivo, y los 7 motivos se reparten así: `Cliente ausente` 102 y `Dirección incorrecta` 49 explican la mitad de los fallos; `Cliente rechazó` 31, `Pedido equivocado` 33 y `Producto dañado` 35 son las devoluciones; `Vehículo averiado` 18 y `Zona de difícil acceso` 40 completan los fallos.

**Decisión:** se **conserva** la columna (no se elimina) porque es el único registro de *por qué* falla una entrega: es la información que permite atacar la causa (por ejemplo, confirmar dirección antes de despachar), y su patrón de vacíos es 100 % explicable por `estado_entrega`. Por la misma razón **no se imputa**.


## 3. Limpieza

Aplicamos las decisiones tomadas: convertir centinelas en faltantes, imputar los faltantes reales con estrategias adecuadas a cada columna y **no** tocar las columnas "no aplica". Al final mostramos el costo de cada decisión.


In [16]:
df_limpio = df.copy()

# Convertimos los valores imposibles en faltantes para imputarlos con criterio.
peso_centinela = df_limpio["peso_kg"].isin([350, 999])
distancia_imposible = df_limpio["distancia_km"].isin([0, 250, 480])

print("Filas con peso_kg centinela (350 o 999):", peso_centinela.sum())
print("Filas con distancia_km imposible (0, 250, 480):", distancia_imposible.sum())

df_limpio.loc[peso_centinela, "peso_kg"] = np.nan
df_limpio.loc[distancia_imposible, "distancia_km"] = np.nan


Filas con peso_kg centinela (350 o 999): 3
Filas con distancia_km imposible (0, 250, 480): 3


**Estrategia de imputación.** Usamos una estrategia distinta según el tipo de variable:

- `canal_compra` es **categórica** → **moda** (el valor más frecuente).
- `peso_kg` es numérica y depende de la categoría de producto → **mediana por grupo** (`categoria_producto`).
- `distancia_km` es numérica y depende del municipio → **mediana por grupo** (`municipio_destino`).

La mediana se prefiere a la media porque no se deja arrastrar por valores extremos.


In [17]:
print("Faltantes en canal_compra antes:", df_limpio["canal_compra"].isna().sum())

moda_canal = df_limpio["canal_compra"].mode()[0]
print("Moda de canal_compra:", moda_canal)

df_limpio["canal_compra"] = df_limpio["canal_compra"].fillna(moda_canal)
print("Faltantes en canal_compra después:", df_limpio["canal_compra"].isna().sum())


Faltantes en canal_compra antes: 223
Moda de canal_compra: App
Faltantes en canal_compra después: 0


**¿Por qué moda?** `canal_compra` es categórica; la moda ("App", 1.746 pedidos) es el valor más frecuente y el más verosímil para reemplazar un canal desconocido sin inventar un valor numérico.


In [18]:
print("Faltantes en peso_kg antes:", df_limpio["peso_kg"].isna().sum(),
      "| mediana general:", round(df_limpio["peso_kg"].median(), 2))

mediana_peso_por_categoria = df_limpio.groupby("categoria_producto")["peso_kg"].median().round(2)
print("\nMediana de peso_kg por categoría:")
print(mediana_peso_por_categoria)

df_limpio["peso_kg"] = df_limpio.groupby("categoria_producto")["peso_kg"].transform(
    lambda s: s.fillna(s.median())
)
print("Faltantes en peso_kg después:", df_limpio["peso_kg"].isna().sum())


Faltantes en peso_kg antes: 250 | mediana general: 3.16

Mediana de peso_kg por categoría:
categoria_producto
Alimentos      4.47
Electrónica    3.72
Farmacia       0.77
Hogar          9.20
Ropa           1.51
Name: peso_kg, dtype: float64
Faltantes en peso_kg después: 0


**¿Por qué mediana por grupo y no mediana general?** El peso depende de la categoría: la mediana por categoría va de **0,77 kg en Farmacia a 9,20 kg en Hogar**, mientras la mediana general es **3,16 kg**. Usar la general le asignaría a cada pedido un peso que no corresponde a su categoría.


In [19]:
print("Faltantes en distancia_km antes:", df_limpio["distancia_km"].isna().sum(),
      "| mediana general:", round(df_limpio["distancia_km"].median(), 2))

mediana_distancia_por_municipio = df_limpio.groupby("municipio_destino")["distancia_km"].median().round(2)
print("\nMediana de distancia_km por municipio:")
print(mediana_distancia_por_municipio)

df_limpio["distancia_km"] = df_limpio.groupby("municipio_destino")["distancia_km"].transform(
    lambda s: s.fillna(s.median())
)
print("Faltantes en distancia_km después:", df_limpio["distancia_km"].isna().sum())


Faltantes en distancia_km antes: 115 | mediana general: 8.7

Mediana de distancia_km por municipio:
municipio_destino
Barbosa        36.85
Bello          10.80
Caldas         22.60
Copacabana     16.55
Envigado        8.80
Girardota      24.10
Itagüí          9.90
La Estrella    15.20
Medellín        6.00
Sabaneta       12.50
Name: distancia_km, dtype: float64
Faltantes en distancia_km después: 0


**¿Por qué mediana por grupo aquí también?** La distancia depende fuertemente del municipio: de **6,0 km en Medellín a 36,9 km en Barbosa**, frente a una mediana general de **8,7 km**. Imputar con la mediana del municipio respeta esa estructura geográfica.


In [20]:
print("Filas antes de la limpieza:", len(df))
print("Filas después de la limpieza:", len(df_limpio))
print("(No se eliminó ninguna fila: se imputaron valores y se justificó conservar las columnas 'no aplica').\n")

print("Faltantes restantes por columna (deben quedar solo las columnas 'no aplica'):")
restantes = df_limpio.isna().sum()
print(restantes[restantes > 0])

costo = pd.DataFrame({
    "columna": ["canal_compra", "peso_kg", "distancia_km"],
    "faltantes_iniciales": [223, 247, 112],
    "centinelas_convertidos": [0, 3, 3],
    "total_imputados": [223, 250, 115],
    "estrategia": ["moda", "mediana por categoria_producto", "mediana por municipio_destino"],
})
print("\nCosto de la limpieza:")
print(costo)


Filas antes de la limpieza: 4100
Filas después de la limpieza: 4100
(No se eliminó ninguna fila: se imputaron valores y se justificó conservar las columnas 'no aplica').

Faltantes restantes por columna (deben quedar solo las columnas 'no aplica'):
tiempo_real_min          209
retraso_min              209
motivo_no_entrega       3792
calificacion_cliente    1108
propina_cop             3173
dtype: int64

Costo de la limpieza:
        columna  faltantes_iniciales  centinelas_convertidos  total_imputados  \
0  canal_compra                  223                       0              223   
1       peso_kg                  247                       3              250   
2  distancia_km                  112                       3              115   

                       estrategia  
0                            moda  
1  mediana por categoria_producto  
2   mediana por municipio_destino  


**Verificación:** tras imputar, `peso_kg`, `canal_compra` y `distancia_km` quedan sin faltantes. Las columnas "no aplica" conservan sus vacíos a propósito. La tabla resume el costo: 223 canal + 250 peso + 115 distancia = **588 valores imputados** sobre 4.100 filas (14,3 %), y **0 filas eliminadas**.


## 4. Preguntas de análisis

Respondemos las cinco preguntas guía del diccionario usando `loc`, filtrado booleano, `sort_values` y `groupby` con agregaciones (siempre incluyendo `count` para saber con cuántos datos se calcula cada promedio).


In [21]:
por_operador = df_limpio.groupby("operador").agg(
    pedidos=("id_pedido", "count"),
    retraso_promedio_min=("retraso_min", "mean"),
    pct_con_retraso=("estado_entrega", lambda s: (s == "Con retraso").mean() * 100),
    pct_fallido_devuelto=("estado_entrega", lambda s: s.isin(["Fallido", "Devuelto"]).mean() * 100),
    calificacion_promedio=("calificacion_cliente", "mean"),
    n_calificaciones=("calificacion_cliente", "count"),
).round(2)

por_operador.sort_values("retraso_promedio_min", ascending=False)


,pedidos,retraso_promedio_min,pct_con_retraso,pct_fallido_devuelto,calificacion_promedio,n_calificaciones
operador,,,,,,
Operador D,667,34.18,61.32,11.09,2.79,503
Operador B,1107,21.61,48.33,5.69,3.46,830
Operador C,1027,20.28,46.84,6.82,3.47,724
Operador A,1299,18.35,44.34,7.78,3.50,935


**Interpretación (Q1).** El operador **D es el peor en las tres dimensiones**: mayor retraso promedio (**34,18 min**), mayor porcentaje con retraso (**61,3 %**), mayor porcentaje fallido/devuelto (**11,1 %**) y la peor calificación (**2,79**). En el otro extremo, **A** es el que menos retrasa (18,35 min) y mejor califica (3,50); **B** es el que menos falla (5,69 %). La tabla tiene `pedidos` y `n_calificaciones` como `count` para saber sobre cuántos casos se calcula cada promedio (D tiene solo 667 pedidos y 503 calificaciones, así que sus promedios pesan menos).


In [22]:
por_lluvia = df_limpio.groupby("lluvia").agg(
    pedidos=("id_pedido", "count"),
    retraso_promedio_min=("retraso_min", "mean"),
    pct_con_retraso=("estado_entrega", lambda s: (s == "Con retraso").mean() * 100),
).round(2)
print("Por lluvia:")
print(por_lluvia)

por_vehiculo_lluvia = df_limpio.groupby(["tipo_vehiculo", "lluvia"]).agg(
    pedidos=("id_pedido", "count"),
    retraso_promedio_min=("retraso_min", "mean"),
).round(2)
print("\nPor tipo de vehículo y lluvia:")
por_vehiculo_lluvia


Por lluvia:
        pedidos  retraso_promedio_min  pct_con_retraso
lluvia                                                
No         2479                 10.93            30.46
Sí         1621                 39.78            76.87

Por tipo de vehículo y lluvia:


pedidos  retraso_promedio_min
tipo_vehiculo       lluvia                               
Bicicleta eléctrica No          176                  5.58
                    Sí          128                 33.66
Camión pequeño      No           26                 21.92
                    Sí           18                 53.44
Moto                No         1924                 10.34
                    Sí         1231                 39.05
Van                 No          353                 16.14
                    Sí          244                 45.65

**Interpretación (Q2).** La lluvia **casi cuadruplica** el retraso promedio: pasa de **10,93 min** sin lluvia a **39,78 min** con lluvia (+28,85 min), y el porcentaje de pedidos con retraso salta de **30,5 % a 76,9 %**. El efecto es grande en **todos** los vehículos: bicicleta (+28,1 min), moto (+28,7), van (+29,5) y camión pequeño (+31,5), aunque el camión y la bicicleta ya parten de peores tiempos. Conclusión: la lluvia golpea a toda la operación por igual, no a un solo vehículo.


In [23]:
# Marcamos temporadas: Black Friday (28-nov-2025) y diciembre.
df_limpio["temporada"] = "Sep–27 nov"
df_limpio.loc[df_limpio["fecha"] == pd.Timestamp("2025-11-28"), "temporada"] = "Black Friday"
df_limpio.loc[df_limpio["fecha"].dt.month == 12, "temporada"] = "Diciembre"

por_temporada = df_limpio.groupby("temporada").agg(
    pedidos=("id_pedido", "count"),
    retraso_promedio_min=("retraso_min", "mean"),
    pct_con_retraso=("estado_entrega", lambda s: (s == "Con retraso").mean() * 100),
).round(2)
print(por_temporada)

print("\nPedidos por mes:")
print(df_limpio["fecha"].dt.to_period("M").value_counts().sort_index())


              pedidos  retraso_promedio_min  pct_con_retraso
temporada                                                   
Black Friday       65                 36.10            69.23
Diciembre        1315                 23.40            49.81
Sep–27 nov       2720                 21.31            47.83

Pedidos por mes:
fecha
2025-09     814
2025-10     862
2025-11    1109
2025-12    1315
Freq: M, Name: count, dtype: int64


**Interpretación (Q3).** El volumen **crece mes a mes**: 814 en septiembre, 862 en octubre, 1.109 en noviembre y 1.315 en diciembre (el día más cargado fue el 25-nov con 92 pedidos). El retraso también sube: en diciembre es **23,40 min** (49,8 % con retraso) frente a **21,31 min** (47,8 %) del resto. El Black Friday (28-nov) tiene el retraso más alto de todos (**36,10 min**, 69,2 % con retraso), pero con solo **65 pedidos**, así que hay que mirarlo con cautela. Conclusión: la temporada alta (nov–dic) sube volumen **y** retraso, por lo que hay que reforzar la operación en esas semanas.


In [24]:
print("Correlación distancia_km vs costo_envio_cop:",
      round(df_limpio["distancia_km"].corr(df_limpio["costo_envio_cop"]), 3))
print("Correlación peso_kg vs costo_envio_cop:",
      round(df_limpio["peso_kg"].corr(df_limpio["costo_envio_cop"]), 3))
print("Costo de envío promedio:", round(df_limpio["costo_envio_cop"].mean(), 0), "COP")

print("\nCosto promedio por rango de distancia:")
df_limpio.groupby(pd.cut(df_limpio["distancia_km"], [0, 5, 10, 20, 40, 100]))["costo_envio_cop"].agg(["count", "mean"]).round(0)


Correlación distancia_km vs costo_envio_cop: 0.83
Correlación peso_kg vs costo_envio_cop: 0.48
Costo de envío promedio: 14026.0 COP

Costo promedio por rango de distancia:


/tmp/ipykernel_544/3694947128.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_limpio.groupby(pd.cut(df_limpio["distancia_km"], [0, 5, 10, 20, 40, 100]))["costo_envio_cop"].agg(["count", "mean"]).round(0)


,count,mean
distancia_km,,
"(0, 5]",575,9390.0
"(5, 10]",1873,11553.0
"(10, 20]",1222,15854.0
"(20, 40]",375,23807.0
"(40, 100]",55,39389.0


**Interpretación (Q4).** La **distancia** es el factor que más explica el costo del envío (r = **0,83**): a mayor distancia, mayor costo. El rango por tramos lo confirma: en 0–5 km el envío cuesta ≈ 9.390 COP y en 40–100 km ≈ 39.389 COP. El **peso** influye menos (r = **0,48**). Distancia y peso entre sí casi no se relacionan (r ≈ 0), así que son dos ejes independientes de precio.


In [25]:
por_estado = df_limpio.groupby("estado_entrega").agg(
    pedidos=("id_pedido", "count"),
    calificacion_promedio=("calificacion_cliente", "mean"),
    n_calificaciones=("calificacion_cliente", "count"),
).round(2)
print("Calificación por estado de entrega:")
print(por_estado)

tramos = pd.cut(df_limpio["retraso_min"], [-200, -30, 0, 15, 30, 60, 300],
                labels=["≤ -30", "-30 a 0", "0 a 15", "15 a 30", "30 a 60", "> 60"])
por_tramo = df_limpio.groupby(tramos)["calificacion_cliente"].agg(["count", "mean"]).round(2)
print("\nCalificación promedio por tramo de retraso (min):")
print(por_tramo)


Calificación por estado de entrega:
                pedidos  calificacion_promedio  n_calificaciones
estado_entrega                                                  
A tiempo           1791                   4.29              1100
Con retraso        2001                   3.03              1649
Devuelto             99                   2.10                78
Fallido             209                   1.06               165

Calificación promedio por tramo de retraso (min):
             count  mean
retraso_min             
≤ -30            8  4.25
-30 a 0        472  4.40
0 a 15         658  4.11
15 a 30        700  3.62
30 a 60        697  2.91
> 60           292  1.71


/tmp/ipykernel_544/1543118209.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  por_tramo = df_limpio.groupby(tramos)["calificacion_cliente"].agg(["count", "mean"]).round(2)


**Interpretación (Q5).** El retraso **explica muy bien** la calificación: la correlación es **−0,76**. La calificación promedio cae de **4,29** (a tiempo) a **3,03** (con retraso), **1,06** (fallido) y **2,10** (devuelto). Por tramos, el descenso es monótono: 4,40 con hasta 0 min de retraso → 4,11 (0–15) → 3,62 (15–30) → 2,91 (30–60) → **1,71** (> 60 min). Los tramos tienen `count` para saber sobre cuántas calificaciones se calcula cada promedio.


## 5. Conclusiones parciales

1. **Operador:** D es el que más retrasa, más falla y peor califica; A es el mejor en tiempo y calificación; B es el que menos falla.
2. **Lluvia:** multiplica el retraso por ~3,6 y casi duplica la tasa de retraso; afecta a todos los vehículos.
3. **Temporada:** el volumen crece hasta diciembre (+62 % vs. septiembre) y el retraso sube; Black Friday es el pico de retraso (muestra pequeña).
4. **Costo:** la distancia manda (r = 0,83); el peso influye menos (r = 0,48).
5. **Satisfacción:** el retraso es el gran enemigo de la calificación (−0,76).

Queda por explorar con gráficas (Entregable 2): cómo se distribuye el retraso, el comportamiento por operador y vehículo, la evolución temporal y las correlaciones entre todas las variables numéricas.
